In [2]:
import pandas as pd

# Define paths to the processed data partitions
processed_dir = "../data/processed"

print("Ingesting clean dataset partitions...")
train_df = pd.read_csv(f"{processed_dir}/train_clean.csv")
val_df = pd.read_csv(f"{processed_dir}/val_clean.csv")
test_df = pd.read_csv(f"{processed_dir}/test_clean.csv")

# Confirm data dimensions match expected counts
print("\n--- Ingestion Verification ---")
print(f"Train Dataset: {train_df.shape[0]:,} rows")
print(f"Val Dataset:   {val_df.shape[0]:,} rows")
print(f"Test Dataset:  {test_df.shape[0]:,} rows")

Ingesting clean dataset partitions...

--- Ingestion Verification ---
Train Dataset: 20,000 rows
Val Dataset:   5,000 rows
Test Dataset:  25,000 rows


# Baseline Sentiment Classification Model
## Section 2.1: Processed Data Ingestion

We initialized the baseline development workspace by loading the standardized data partitions generated during our preprocessing workflow.

* **Partition Verification:** The exact row configurations perfectly match our upstream outputs—20,000 records for model training, 5,000 records for validation tracking, and 25,000 records reserved for final holdout benchmarking.
* **Separation of Concerns:** By feeding identical text splits into both this traditional pipeline and our subsequent transformer model, we maintain a strict, controlled environment for experimental comparison.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Build an integrated feature extraction and classification pipeline
print("Constructing TF-IDF + Logistic Regression Baseline Pipeline...")
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

# Train the model pipeline on the training split
print("Training model on 20,000 clean review strings...")
baseline_pipeline.fit(train_df['cleaned_text'], train_df['label'])

# Evaluate performance using the validation split
print("Evaluating on validation data...")
val_preds = baseline_pipeline.predict(val_df['cleaned_text'])

# Display initial validation metrics
print("\n--- Baseline Validation Performance ---")
print(f"Overall Accuracy: {accuracy_score(val_df['label'], val_preds):.4f}")
print("\nClassification Report:\n")
print(classification_report(val_df['label'], val_preds, target_names=['Negative', 'Positive']))

Constructing TF-IDF + Logistic Regression Baseline Pipeline...
Training model on 20,000 clean review strings...
Evaluating on validation data...

--- Baseline Validation Performance ---
Overall Accuracy: 0.8932

Classification Report:

              precision    recall  f1-score   support

    Negative       0.90      0.89      0.89      2500
    Positive       0.89      0.90      0.89      2500

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000



## Section 2.2: Baseline Model Performance & Validation Analysis

We successfully trained our baseline pipeline using TF-IDF text vectorization and a Logistic Regression classifier. The evaluation on our isolated validation partition yielded the following insights:

* **High Performance Benchmark:** The model achieved an overall classification accuracy of 89.32%, establishing a strong performance threshold for a traditional linear approach.
* **Balanced Classification:** Precision, recall, and F1-scores are evenly distributed across both sentiment classes (0.89 to 0.90). The model does not show any bias toward positive or negative reviews, which directly reflects the benefits of our perfectly balanced training partition.
* **Feature Efficacy:** The metrics confirm that word frequency patterns and basic two-word sequences (bi-grams) capture a substantial amount of the sentiment signal in movie reviews, even without modeling complex sequence dependencies.

In [4]:
import joblib
from sklearn.metrics import classification_report, accuracy_score

# Generate predictions on the completely unseen holdout test set
print("Running final evaluation on the holdout test partition (25,000 rows)...")
test_preds = baseline_pipeline.predict(test_df['cleaned_text'])

# Print final test metrics
print("\n--- Final Baseline Test Performance ---")
print(f"Test Accuracy: {accuracy_score(test_df['label'], test_preds):.4f}")
print("\nFinal Test Classification Report:\n")
print(classification_report(test_df['label'], test_preds, target_names=['Negative', 'Positive']))

# Serialize the trained baseline pipeline to disk
model_path = "../models/baseline_logreg.pkl"
print(f"Saving trained pipeline artifact to: {model_path}")
joblib.dump(baseline_pipeline, model_path)
print("Serialization complete.")

Running final evaluation on the holdout test partition (25,000 rows)...

--- Final Baseline Test Performance ---
Test Accuracy: 0.8892

Final Test Classification Report:

              precision    recall  f1-score   support

    Negative       0.89      0.88      0.89     12500
    Positive       0.88      0.90      0.89     12500

    accuracy                           0.89     25000
   macro avg       0.89      0.89      0.89     25000
weighted avg       0.89      0.89      0.89     25000

Saving trained pipeline artifact to: ../models/baseline_logreg.pkl
Serialization complete.


## Section 2.3: Holdout Test Benchmarking & Model Serialization

The baseline pipeline underwent a final evaluation against the 25,000-row holdout test set to establish our official performance benchmark.

### Operational Insights:
* **High Generalizability:** The model achieved a final test accuracy of 88.92%. The minimal variance between our validation score (89.32%) and this test score confirms the model generalizes well and is free from overfitting.
* **Metric Consistency:** Precision and recall figures held steady between 0.88 and 0.90 across both categories. This verifies consistent class performance on completely unseen review distributions.
* **Artifact Preservation:** The trained pipeline was serialized directly to `../models/baseline_logreg.pkl`. This standalone binary artifact is now ready to be loaded directly by our web interface without requiring any upstream training steps.